# 03 — Model training and validation

**Purpose.** Build the internal company-level validation split, train the seven main models, select hyperparameters with validation PR-AUC, and save validation outputs.

**Inputs.** `data/processed/train.csv` from Notebook 02.

**Outputs.** Validation metrics, classification reports, validation predictions, and selected model settings.

**What you will learn.** How the model comparison is fixed before the untouched final test set is used.

**Run first.** Run Notebooks 01-02 first.

## Imports and paths

I keep the package imports and project-relative folders together here. The path logic works from either the project root or the `notebooks/` folder, so the notebook can be rerun without manual path edits.


In [1]:
from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

current_path = Path.cwd().resolve()
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

assert project_root.name == "ml-finance-bankruptcy-analysis", (
    f"Unexpected project root: {project_root}"
)

DATA_DIR = project_root / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = project_root / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
REPORT_DIR = OUTPUTS_DIR / "report"
PAPER_TABLES_DIR = OUTPUTS_DIR / "paper" / "tables"
PAPER_FIGURES_DIR = OUTPUTS_DIR / "paper" / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORT_DIR,
    PAPER_TABLES_DIR,
    PAPER_FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


## Project constants

These constants define the raw column names, label values, split sizes, model order, and output folders used later in the notebook. Keeping them in one place makes the modelling choices easier to audit.


In [2]:
RAW_DATA_PATH = RAW_DATA_DIR / "american_bankruptcy.csv"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]

EXPECTED_MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "L1 Regularized Logistic Regression",
    "L2 Regularized Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
]

PREDICTION_TABLE_COLUMNS = [
    "model",
    COMPANY_COLUMN,
    YEAR_COLUMN,
    "actual_failed",
    "predicted_failed",
    "probability_failed",
]
PREDICTION_PROBABILITY_FLOAT_FORMAT = "%.11f"


## Feature names and descriptions

The dataset uses the original `X1`-`X18` variable names. This dictionary adds readable labels for tables, plots, coefficients, and feature-importance outputs while preserving the original column codes.


In [3]:
FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]


## Figure style helpers

These helpers keep the figures visually consistent across notebooks. They only control formatting, labels, and saving; the plotted data are created in the analysis cells below.


In [4]:
MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Random Forest": "#59a14f",
    "Gradient Boosting": "#f28e2b",
}
OUTCOME_COLORS = {"detected": "#59a14f", "missed": "#e15759", "false_alarm": "#f28e2b"}
DIRECTION_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}
METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}
BASELINE_COLOR = "#8c8c8c"


def apply_project_style() -> None:
    """Apply the common Matplotlib style used across project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Add consistent grid and tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a Matplotlib figure with the project's standard export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Data validation and feature helpers

These checks make the required columns, target labels, and predictor matrix explicit before modelling. I reuse them so a later notebook fails early if an earlier file is missing or malformed.


In [5]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Confirm that the raw dataset has the identifier, year, and target columns."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Confirm that the raw target contains only the expected alive/failed labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables from the raw dataset, excluding identifiers."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while keeping identifiers and target out of X."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictor matrix and binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()


## Company split helpers

The split functions work at the company level. This avoids leakage from the same firm appearing in both training and evaluation samples.


In [6]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        """Return row counts, company counts, and target mix for one split."""
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary


## Evaluation helpers

The same evaluation functions are used for validation and final testing, so every model is compared with the same accuracy, balanced accuracy, failed-class precision/recall/F1, ROC-AUC, PR-AUC, and confusion-count definitions.


In [7]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the common classification metrics for one fitted model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def create_classification_report_table(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Convert scikit-learn's classification report into a flat table."""
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["alive", "failed"],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({"model": model_name, "class_or_average": label, **metrics})
    return pd.DataFrame(rows)


def create_prediction_table(
    data: pd.DataFrame,
    model_name: str,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> pd.DataFrame:
    """Create an identifier-preserving prediction table for one model."""
    table = pd.DataFrame(
        {
            "model": model_name,
            COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(),
            YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(),
            "actual_failed": data[TARGET_COLUMN].to_numpy(),
            "predicted_failed": y_pred,
            "probability_failed": probability_failed,
        }
    )
    return table[PREDICTION_TABLE_COLUMNS]


def save_prediction_table(predictions: pd.DataFrame, output_path: Path) -> None:
    """Write prediction probabilities with deterministic decimal formatting."""
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(
        output_path,
        index=False,
        float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT,
    )


## Model builders and selection helpers

The model builders collect the exact classifier settings in one readable block. I repeat them across notebooks so each file can still be rerun on its own, even though not every notebook calls every helper.


In [8]:
def build_majority_class_baseline() -> DummyClassifier:
    """Build the benchmark model that always predicts the most frequent class."""
    return DummyClassifier(strategy="most_frequent")


def build_logistic_pipeline(
    penalty: str = "l2",
    c_value: float = 1.0,
    class_weight: str | dict | None = "balanced",
) -> Pipeline:
    """Build a scaled Logistic Regression pipeline with the original settings."""
    if penalty not in {"l1", "l2"}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")

    model = LogisticRegression(
        C=c_value,
        penalty=penalty,
        solver="saga",
        class_weight=class_weight,
        max_iter=5_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])


def build_interpretable_logit() -> Pipeline:
    """Build the fixed Logistic Regression benchmark used for interpretation."""
    return build_logistic_pipeline(penalty="l2", c_value=1.0)


def select_regularized_logit(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
    penalty: str,
    c_grid: list[float],
) -> tuple[Pipeline, float, float]:
    """Select L1 or L2 Logistic Regression by validation PR-AUC."""
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None or best_c is None:
        raise RuntimeError("No Logistic Regression model was selected.")
    return best_model, best_c, best_score


def build_decision_tree(max_depth: int | None = 3, min_samples_leaf: int = 100) -> DecisionTreeClassifier:
    """Build a class-weighted Decision Tree with the source project settings."""
    return DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def build_random_forest(
    n_estimators: int = 300,
    max_depth: int | None = 5,
    min_samples_leaf: int = 50,
    max_features: str | None = "sqrt",
) -> RandomForestClassifier:
    """Build a Random Forest using the original class weights and seed."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_gradient_boosting(
    n_estimators: int = 150,
    learning_rate: float = 0.05,
    max_depth: int = 2,
    min_samples_leaf: int = 100,
) -> GradientBoostingClassifier:
    """Build a Gradient Boosting classifier with the source project settings."""
    return GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )


def select_decision_tree(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[DecisionTreeClassifier, dict[str, int], float]:
    """Search the original Decision Tree grid and keep the best PR-AUC model."""
    grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid["max_depth"], grid["min_samples_leaf"]):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Decision Tree model was selected.")
    return best_model, best_params, best_score


def select_random_forest(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[RandomForestClassifier, dict[str, object], float]:
    """Search the original Random Forest grid and keep the best PR-AUC model."""
    grid = {
        "n_estimators": [300],
        "max_depth": [4, 6, None],
        "min_samples_leaf": [50, 100],
        "max_features": ["sqrt"],
    }
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(
        grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"], grid["max_features"]
    ):
        candidate = build_random_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
        )
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Random Forest model was selected.")
    return best_model, best_params, best_score


def select_gradient_boosting(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[GradientBoostingClassifier, dict[str, object], float]:
    """Search the original Gradient Boosting grid using balanced sample weights."""
    grid = {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [100],
    }
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
        grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["min_samples_leaf"]
    ):
        candidate = build_gradient_boosting(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Gradient Boosting model was selected.")
    return best_model, best_params, best_score


## Model reconstruction helpers

These helpers rebuild models from saved parameter tables instead of relying on hidden model binaries. That keeps the replication path transparent.


In [9]:
def parse_selected_parameters(value: object) -> dict[str, object]:
    """Parse the saved parameter string from `model_specification.csv`."""
    if pd.isna(value) or value == "":
        return {}

    text = str(value)
    if text.startswith("{"):
        return ast.literal_eval(text)

    params: dict[str, object] = {}
    for piece in text.split(","):
        if "=" in piece:
            key, val = piece.strip().split("=", 1)
            params[key] = float(val) if key == "C" else val
    return params


def build_model_from_spec(model_name: str, selected_parameters: object):
    """Recreate one model from its saved specification without using model binaries."""
    params = parse_selected_parameters(selected_parameters)
    if model_name == "Majority-class baseline":
        return build_majority_class_baseline()
    if model_name == "Logistic Regression":
        return build_interpretable_logit()
    if model_name == "L1 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l1", c_value=params["C"])
    if model_name == "L2 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l2", c_value=params["C"])
    if model_name == "Decision Tree":
        return build_decision_tree(**params)
    if model_name == "Random Forest":
        return build_random_forest(**params)
    if model_name == "Gradient Boosting":
        return build_gradient_boosting(**params)
    raise ValueError(f"Unknown model: {model_name}")


## Load the outer-training sample

Only the training file from Notebook 02 is loaded here. The final test file is deliberately left untouched during model selection.


In [10]:
train_full = pd.read_csv(TRAIN_DATA_PATH)
display(train_full.head())
display(pd.DataFrame({"rows": [len(train_full)], "companies": [train_full[COMPANY_COLUMN].nunique()], "failure_rate": [train_full[TARGET_COLUMN].mean()]}))

,company_name,year,failed,X1,X2,X3,X4,X5,X6,X7,...,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18
0,C_3,1999,0,9.757,19.796,0.667,-0.265,5.494,-2.207,3.924,...,29.370,13.986,5.974,-0.932,9.574,2.804,-6.375,29.370,8.778,29.635
1,C_3,2000,0,7.884,16.506,0.700,0.672,4.078,-0.808,3.244,...,25.367,11.608,4.875,-0.028,8.861,2.278,-7.184,25.367,7.153,24.695
2,C_3,2001,0,6.494,15.700,0.761,0.381,3.488,-1.738,2.677,...,24.051,8.635,3.873,-0.380,8.351,2.045,-8.922,24.051,5.918,23.670
3,C_3,2002,0,5.938,12.919,0.355,0.711,2.816,0.084,2.465,...,20.087,7.850,2.546,0.356,7.168,2.481,-8.816,20.087,5.027,19.376
4,C_3,2004,0,5.807,12.018,0.160,1.614,2.704,1.345,2.504,...,19.833,6.245,0.222,1.454,7.815,3.222,-8.974,19.833,3.580,18.219


,rows,companies,failure_rate
0,62988,7176,0.064187


## Create the internal company-level validation split

This split is used for model selection. The final test set is not loaded in this notebook.

In [11]:
model_train, validation = create_company_level_split(
    train_full,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)
assert check_no_company_overlap(model_train, validation), "Internal split contains company overlap."
assert len(model_train) + len(validation) == len(train_full), "Internal split must preserve training rows."

x_train, y_train = split_features_target(model_train)
x_valid, y_valid = split_features_target(validation)

internal_split_summary = pd.DataFrame(
    {
        "sample": ["model_train", "validation"],
        "rows": [len(model_train), len(validation)],
        "companies": [model_train[COMPANY_COLUMN].nunique(), validation[COMPANY_COLUMN].nunique()],
        "failed_rows": [int(y_train.sum()), int(y_valid.sum())],
        "failure_rate": [y_train.mean(), y_valid.mean()],
    }
)
display(internal_split_summary)

,sample,rows,companies,failed_rows,failure_rate
0,model_train,50323,5740,3213,0.063848
1,validation,12665,1436,830,0.065535


## Prepare result containers

These dictionaries and lists collect the fitted validation models, selected settings, metric rows, classification reports, and row-level predictions.


In [12]:
models = {}
model_specs = []

metric_rows = []
report_tables = []
prediction_tables = []

## Majority-class baseline

This simple baseline always predicts the majority class. It gives a useful reference point for accuracy in an imbalanced bankruptcy sample.


In [13]:
majority_model = build_majority_class_baseline()
majority_model.fit(x_train, y_train)
models["Majority-class baseline"] = majority_model
model_specs.append(
    {
        "model": "Majority-class baseline",
        "model_type": "DummyClassifier",
        "model_family": "baseline",
        "selected_parameters": "",
        "selection_metric": "",
        "validation_pr_auc_during_selection": "",
    }
)
display(pd.DataFrame([model_specs[-1]]))

,model,model_type,model_family,selected_parameters,selection_metric,validation_pr_auc_during_selection
0,Majority-class baseline,DummyClassifier,baseline,,,


## Fixed Logistic Regression benchmark

This is the main interpretable linear benchmark: standardized predictors, L2 penalty, and balanced class weights.


In [14]:
logit = build_interpretable_logit()
logit.fit(x_train, y_train)
models["Logistic Regression"] = logit
model_specs.append(
    {
        "model": "Logistic Regression",
        "model_type": "LogisticRegression",
        "model_family": "logistic_regression",
        "selected_parameters": "penalty=l2, C=1.0",
        "selection_metric": "fixed benchmark",
        "validation_pr_auc_during_selection": "",
    }
)
display(pd.DataFrame([model_specs[-1]]))

,model,model_type,model_family,selected_parameters,selection_metric,validation_pr_auc_during_selection
0,Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=1.0",fixed benchmark,


## L1 Logistic Regression candidates

The L1 grid tests sparse logistic specifications. The selected value is the one with the highest validation PR-AUC.


In [15]:
l1_rows = []
best_l1_model, best_l1_c, best_l1_score = None, None, -1.0
for c_value in LOGISTIC_C_GRID:
    candidate = build_logistic_pipeline(penalty="l1", c_value=c_value)
    candidate.fit(x_train, y_train)
    score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
    l1_rows.append({"penalty": "l1", "C": c_value, "validation_pr_auc": score})
    if score > best_l1_score:
        best_l1_model, best_l1_c, best_l1_score = candidate, c_value, score

models["L1 Regularized Logistic Regression"] = best_l1_model
model_specs.append(
    {
        "model": "L1 Regularized Logistic Regression",
        "model_type": "LogisticRegression",
        "model_family": "logistic_regression",
        "selected_parameters": f"penalty=l1, C={best_l1_c}",
        "selection_metric": "validation PR-AUC",
        "validation_pr_auc_during_selection": best_l1_score,
    }
)
assert best_l1_c in LOGISTIC_C_GRID, "Selected L1 C must come from the declared grid."
l1_candidate_table = pd.DataFrame(l1_rows).sort_values("validation_pr_auc", ascending=False)
display(l1_candidate_table)

,penalty,C,validation_pr_auc
1,l1,0.10,0.148295
2,l1,1.00,0.147452
3,l1,10.00,0.147361
0,l1,0.01,0.146259


## L2 Logistic Regression candidates

The L2 grid tests different shrinkage strengths while keeping all financial predictors in the model.


In [16]:
l2_rows = []
best_l2_model, best_l2_c, best_l2_score = None, None, -1.0
for c_value in LOGISTIC_C_GRID:
    candidate = build_logistic_pipeline(penalty="l2", c_value=c_value)
    candidate.fit(x_train, y_train)
    score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
    l2_rows.append({"penalty": "l2", "C": c_value, "validation_pr_auc": score})
    if score > best_l2_score:
        best_l2_model, best_l2_c, best_l2_score = candidate, c_value, score

models["L2 Regularized Logistic Regression"] = best_l2_model
model_specs.append(
    {
        "model": "L2 Regularized Logistic Regression",
        "model_type": "LogisticRegression",
        "model_family": "logistic_regression",
        "selected_parameters": f"penalty=l2, C={best_l2_c}",
        "selection_metric": "validation PR-AUC",
        "validation_pr_auc_during_selection": best_l2_score,
    }
)
assert best_l2_c in LOGISTIC_C_GRID, "Selected L2 C must come from the declared grid."
l2_candidate_table = pd.DataFrame(l2_rows).sort_values("validation_pr_auc", ascending=False)
display(l2_candidate_table)

,penalty,C,validation_pr_auc
1,l2,0.10,0.148730
2,l2,1.00,0.147629
3,l2,10.00,0.147383
0,l2,0.01,0.144861


## Decision Tree candidates

The tree grid varies depth and minimum leaf size, then keeps the specification with the strongest validation ranking performance.


In [17]:
decision_tree_grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
dt_rows = []
best_dt_model, best_dt_params, best_dt_score = None, None, -1.0
for max_depth, min_samples_leaf in product(decision_tree_grid["max_depth"], decision_tree_grid["min_samples_leaf"]):
    candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
    candidate.fit(x_train, y_train)
    score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
    dt_rows.append({"max_depth": max_depth, "min_samples_leaf": min_samples_leaf, "validation_pr_auc": score})
    if score > best_dt_score:
        best_dt_model = candidate
        best_dt_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
        best_dt_score = score

models["Decision Tree"] = best_dt_model
model_specs.append(
    {
        "model": "Decision Tree",
        "model_type": "DecisionTreeClassifier",
        "model_family": "tree_based",
        "selected_parameters": str(best_dt_params),
        "selection_metric": "validation PR-AUC",
        "validation_pr_auc_during_selection": best_dt_score,
    }
)
dt_candidate_table = pd.DataFrame(dt_rows).sort_values("validation_pr_auc", ascending=False)
display(dt_candidate_table)

,max_depth,min_samples_leaf,validation_pr_auc
9,5,50,0.146024
10,5,100,0.143485
11,5,200,0.143073
8,4,200,0.129338
6,4,50,0.129148
7,4,100,0.129148
3,3,50,0.105887
4,3,100,0.105887
5,3,200,0.105887
0,2,50,0.100468


## Random Forest candidates

The forest grid compares tree complexity while keeping the seed, class weights, and feature-sampling rule fixed.


In [18]:
random_forest_grid = {
    "n_estimators": [300],
    "max_depth": [4, 6, None],
    "min_samples_leaf": [50, 100],
    "max_features": ["sqrt"],
}
rf_rows = []
best_rf_model, best_rf_params, best_rf_score = None, None, -1.0
for n_estimators, max_depth, min_samples_leaf, max_features in product(
    random_forest_grid["n_estimators"],
    random_forest_grid["max_depth"],
    random_forest_grid["min_samples_leaf"],
    random_forest_grid["max_features"],
):
    candidate = build_random_forest(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features)
    candidate.fit(x_train, y_train)
    score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
    rf_rows.append({"n_estimators": n_estimators, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf, "max_features": max_features, "validation_pr_auc": score})
    if score > best_rf_score:
        best_rf_model = candidate
        best_rf_params = {"n_estimators": n_estimators, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf, "max_features": max_features}
        best_rf_score = score

models["Random Forest"] = best_rf_model
model_specs.append(
    {
        "model": "Random Forest",
        "model_type": "RandomForestClassifier",
        "model_family": "tree_based",
        "selected_parameters": str(best_rf_params),
        "selection_metric": "validation PR-AUC",
        "validation_pr_auc_during_selection": best_rf_score,
    }
)
rf_candidate_table = pd.DataFrame(rf_rows).sort_values("validation_pr_auc", ascending=False)
display(rf_candidate_table)

,n_estimators,max_depth,min_samples_leaf,max_features,validation_pr_auc
4,300,NaN,50,sqrt,0.154216
5,300,NaN,100,sqrt,0.154011
3,300,6.0,100,sqrt,0.151104
1,300,4.0,100,sqrt,0.149157
0,300,4.0,50,sqrt,0.147874
2,300,6.0,50,sqrt,0.147433


## Gradient Boosting candidates

Boosting is fitted with balanced sample weights. The selected specification follows the same validation PR-AUC rule used for the other tuned models.


In [19]:
gradient_boosting_grid = {
    "n_estimators": [100, 150],
    "learning_rate": [0.03, 0.05],
    "max_depth": [2, 3],
    "min_samples_leaf": [100],
}
gb_rows = []
gb_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
best_gb_model, best_gb_params, best_gb_score = None, None, -1.0
for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
    gradient_boosting_grid["n_estimators"],
    gradient_boosting_grid["learning_rate"],
    gradient_boosting_grid["max_depth"],
    gradient_boosting_grid["min_samples_leaf"],
):
    candidate = build_gradient_boosting(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, min_samples_leaf=min_samples_leaf)
    candidate.fit(x_train, y_train, sample_weight=gb_sample_weight)
    score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
    gb_rows.append({"n_estimators": n_estimators, "learning_rate": learning_rate, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf, "validation_pr_auc": score})
    if score > best_gb_score:
        best_gb_model = candidate
        best_gb_params = {"n_estimators": n_estimators, "learning_rate": learning_rate, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
        best_gb_score = score

models["Gradient Boosting"] = best_gb_model
model_specs.append(
    {
        "model": "Gradient Boosting",
        "model_type": "GradientBoostingClassifier",
        "model_family": "tree_based",
        "selected_parameters": str(best_gb_params),
        "selection_metric": "validation PR-AUC",
        "validation_pr_auc_during_selection": best_gb_score,
    }
)
gb_candidate_table = pd.DataFrame(gb_rows).sort_values("validation_pr_auc", ascending=False)
display(gb_candidate_table)

,n_estimators,learning_rate,max_depth,min_samples_leaf,validation_pr_auc
7,150,0.05,3,100,0.174674
3,100,0.05,3,100,0.170132
6,150,0.05,2,100,0.168995
2,100,0.05,2,100,0.166435
4,150,0.03,2,100,0.164951
5,150,0.03,3,100,0.164949
0,100,0.03,2,100,0.162412
1,100,0.03,3,100,0.161024


## Selected model settings

This table is the audit trail for the seven specifications carried forward to final testing.


In [20]:
assert list(models.keys()) == EXPECTED_MODEL_ORDER, "Unexpected model order or names."
model_specification = pd.DataFrame(model_specs)
display(model_specification)

,model,model_type,model_family,selected_parameters,selection_metric,validation_pr_auc_during_selection
0,Majority-class baseline,DummyClassifier,baseline,,,
1,Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=1.0",fixed benchmark,
2,L1 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l1, C=0.1",validation PR-AUC,0.148295
3,L2 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=0.1",validation PR-AUC,0.14873
4,Decision Tree,DecisionTreeClassifier,tree_based,"{'max_depth': 5, 'min_samples_leaf': 50}",validation PR-AUC,0.146024
5,Random Forest,RandomForestClassifier,tree_based,"{'n_estimators': 300, 'max_depth': None, 'min_...",validation PR-AUC,0.154216
6,Gradient Boosting,GradientBoostingClassifier,tree_based,"{'n_estimators': 150, 'learning_rate': 0.05, '...",validation PR-AUC,0.174674


## Validation evaluation for all seven models

After selection, all seven models are evaluated on the same internal validation sample with the same metric function.


In [21]:
metric_rows, report_tables, prediction_tables = [], [], []
for model_name, model in models.items():
    y_pred = model.predict(x_valid)
    probability_failed = get_probability_failed(model, x_valid)
    assert len(probability_failed) == len(validation), f"{model_name} probability length mismatch."
    assert pd.Series(probability_failed).between(0, 1).all(), f"{model_name} probabilities outside [0, 1]."
    metric_rows.append(evaluate_binary_classifier(model_name, y_valid, y_pred, probability_failed))
    report_tables.append(create_classification_report_table(model_name, y_valid, y_pred))
    prediction_tables.append(create_prediction_table(validation, model_name, y_pred, probability_failed))

validation_model_comparison = pd.DataFrame(metric_rows)
validation_classification_reports = pd.concat(report_tables, ignore_index=True)
validation_predictions = pd.concat(prediction_tables, ignore_index=True)

assert list(validation_model_comparison["model"]) == EXPECTED_MODEL_ORDER, "Validation model order changed."
assert validation_predictions.groupby("model").size().eq(len(validation)).all(), "Each model must produce one validation prediction per row."
for _, row in validation_model_comparison.iterrows():
    assert row[["true_negative", "false_positive", "false_negative", "true_positive"]].sum() == len(validation), f"Confusion counts inconsistent for {row.model}."

display(validation_model_comparison)

,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed,true_negative,false_positive,false_negative,true_positive
0,Majority-class baseline,0.934465,0.500000,0.500000,0.065535,0.000000,0.000000,0.000000,11835,0,830,0
1,Logistic Regression,0.366996,0.589600,0.671009,0.147629,0.081713,0.845783,0.149029,3946,7889,128,702
2,L1 Regularized Logistic Regression,0.367075,0.592444,0.671667,0.148295,0.082209,0.851807,0.149947,3942,7893,123,707
3,L2 Regularized Logistic Regression,0.366285,0.592021,0.672404,0.148730,0.082114,0.851807,0.149788,3932,7903,123,707
4,Decision Tree,0.604737,0.587410,0.643907,0.146024,0.092028,0.567470,0.158373,7188,4647,359,471
5,Random Forest,0.816502,0.608852,0.702748,0.154216,0.145636,0.369880,0.208986,10034,1801,523,307
6,Gradient Boosting,0.677852,0.627652,0.691807,0.174674,0.112726,0.569880,0.188221,8112,3723,357,473


## Save validation outputs

These CSV files are the canonical validation records used by the final-test notebook, threshold analysis, PCA extension, and paper assets.


In [22]:
validation_model_comparison.to_csv(TABLES_DIR / "validation_model_comparison.csv", index=False)
validation_classification_reports.to_csv(TABLES_DIR / "validation_classification_reports.csv", index=False)
model_specification.to_csv(TABLES_DIR / "model_specification.csv", index=False)
save_prediction_table(validation_predictions, TABLES_DIR / "validation_predictions.csv")

saved_outputs = pd.DataFrame(
    {
        "output": [
            "outputs/tables/validation_model_comparison.csv",
            "outputs/tables/validation_classification_reports.csv",
            "outputs/tables/model_specification.csv",
            "outputs/tables/validation_predictions.csv",
        ],
        "exists": [
            (TABLES_DIR / "validation_model_comparison.csv").exists(),
            (TABLES_DIR / "validation_classification_reports.csv").exists(),
            (TABLES_DIR / "model_specification.csv").exists(),
            (TABLES_DIR / "validation_predictions.csv").exists(),
        ],
    }
)
display(saved_outputs)

,output,exists
0,outputs/tables/validation_model_comparison.csv,True
1,outputs/tables/validation_classification_repor...,True
2,outputs/tables/model_specification.csv,True
3,outputs/tables/validation_predictions.csv,True


## Summary

- Built an internal company-level validation split from the outer-training data; the final test file is not used here.
- Trained and displayed the seven model specifications step by step: baseline, fixed Logistic Regression, L1 Logistic Regression, L2 Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.
- Selected tuned models by validation PR-AUC and saved the chosen settings in `outputs/tables/model_specification.csv`.
- Saved validation metrics, classification reports, and row-level predictions for the later threshold, PCA, and paper-output notebooks.
